# GlucoLens: Explainable Machine Learning for Early Prediction of Diabetes Risk

**Course project, B.Tech CSE (AI/ML).** We use routine clinical measurements to:

1. **Classify** whether a person has diabetes (screening). Dataset: *Pima Indians Diabetes*.
2. **Predict** how much the disease progresses in one year (a continuous value). Dataset: *Diabetes Progression* (Efron et al., 2004).

We also **explain** which clinical factors drive the predictions.

## Rubric map

| # | Rubric item | Marks | Where it is covered |
|---|---|---|---|
| 1 | Problem Understanding & Familiarization | 4 | **Section A** |
| 2 | Exploratory Data Analysis & Insights | 4 | **Section B** |
| 3 | Unit 1 / Unit 2 Topic Implementation (Regression **and** Classification) | 4 | **Sections C, D, E** |
| 4 | Comparative Performance Analysis | 4 | **Section F** |
| 5 | Result Interpretation & Presentation | 4 | **Sections G, H** |

> How to run: select the project's `.venv` kernel, then **Restart & Run All**. The first run downloads the Pima data from OpenML (internet needed once); after that it is read from `data/`.

## Setup

**What:** import the libraries, fix the random seed, and create the `data/` and `figures/` folders.

**Why:** a single `RANDOM_STATE` makes every random step (train/test split, cross-validation folds, random forests) give the same result on every run, so our numbers are reproducible. The folders hold the saved datasets and the figures for the slides.

**Formula:** none in this cell.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import fetch_openml, load_diabetes

RANDOM_STATE = 42          # the ONE seed used everywhere in this notebook
np.random.seed(RANDOM_STATE)

os.makedirs("data", exist_ok=True)      # saved datasets (offline after first run)
os.makedirs("figures", exist_ok=True)   # saved plots for the report/slides

sns.set_theme(style="whitegrid")
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

pandas 3.0.6 | numpy 2.4.6 | scikit-learn 1.9.1


# Section A: Problem Understanding & Familiarization *(Rubric 1)*

## A.1 What is diabetes?

**Diabetes mellitus** is a long-term condition in which blood glucose (sugar) stays too high. Glucose levels are controlled by **insulin**, a hormone made by the pancreas that lets cells absorb glucose from the blood.

| | **Type 1 diabetes** | **Type 2 diabetes** |
|---|---|---|
| Cause | The immune system destroys the insulin-producing cells, so the body makes little or no insulin | The body becomes **resistant** to insulin, and over time the pancreas cannot make enough to compensate |
| Typical onset | Usually childhood or young adulthood | Usually adulthood (increasingly seen in younger people) |
| Share of cases | Minority | The large majority of cases |
| Link to lifestyle | Not preventable by lifestyle | Strongly linked to obesity, inactivity and diet; often **preventable or delayable** |

Both of our datasets are about adults, so the patterns they contain are mostly those of **Type 2 diabetes**.

## A.2 Why early detection matters

- Type 2 diabetes develops slowly and can go **unnoticed for years**, while high glucose silently damages the eyes, kidneys, nerves, heart and blood vessels.
- Many people are diagnosed only after complications have started.
- If people at high risk are found early, lifestyle changes and treatment can **delay or prevent** the disease and its complications.
- A cheap screening model built on routine measurements could help decide **who should get a confirmatory test** first.

## A.3 Known clinical risk factors

| Risk factor | Why it matters |
|---|---|
| **High blood glucose** | The defining feature of diabetes; high fasting or post-meal glucose is the main diagnostic sign |
| **High BMI / obesity** | Excess body fat increases insulin resistance |
| **Age** | Risk rises with age as insulin sensitivity and pancreatic function decline |
| **Family history** | Genes inherited from relatives raise the risk |
| **High blood pressure** | Often occurs together with insulin resistance; also adds to the damage diabetes does to blood vessels |

We will check later whether our models "discover" the same factors (Section G).

## A.4 Why explainability matters in healthcare

A model that only outputs "diabetic: yes/no" is not enough in medicine:

- **Trust:** doctors will only act on a prediction if they can see *why* the model made it.
- **Accountability:** if a decision affects a patient, someone must be able to justify it.
- **Catching model errors:** explanations show whether a model relies on sensible clinical signals or on artefacts in the data (for example, missing values recorded as `0`).
- **Actionable advice:** knowing which *modifiable* factors (like BMI) push risk up tells us what a patient can change.

This is why every model in this notebook is followed by an explanation (coefficients, permutation importance, SHAP).

## A.5 The two machine-learning tasks

| | **Task 1: Classification** | **Task 2: Regression** |
|---|---|---|
| Dataset | Pima Indians Diabetes | Diabetes Progression (Efron et al., 2004) |
| Inputs $\mathbf{x}$ | 8 clinical measurements | 10 baseline measurements |
| Target $y$ | `Outcome`: 1 = diabetic, 0 = not diabetic | `target`: disease progression one year after baseline |
| Type of target | **Categorical** (two classes) | **Continuous** (a number on a scale) |
| Question answered | "Does this person have diabetes?" | "How much will this patient's disease progress?" |

**Why each task type fits its target.**

- `Outcome` can only be 0 or 1, so we need a **classifier**. Most classifiers first estimate a probability and then apply a threshold $t$ (0.5 by default):

$$\hat{p} = P(y = 1 \mid \mathbf{x}), \qquad \hat{y} = \begin{cases} 1 & \text{if } \hat{p} \ge t \\ 0 & \text{otherwise} \end{cases}$$

- The progression score can take any value in a range, so we need a **regression** model that outputs a real number:

$$\hat{y} = f(\mathbf{x}), \qquad \hat{y} \in \mathbb{R}$$

where $\mathbf{x}$ is the vector of a patient's features, $y$ is the true target, $\hat{y}$ is the model's prediction, $\hat{p}$ is the predicted probability of diabetes, and $f$ is the function the model learns from data.

## A.6 Feature dictionary

### Pima Indians Diabetes (classification)

All 768 patients are **women aged 21 or older of Pima Indian heritage** (Arizona, USA).

| Feature | Unit | Clinical meaning | Modifiable? |
|---|---|---|---|
| `Pregnancies` | count | Number of times pregnant (pregnancy can trigger gestational diabetes, a risk factor for Type 2) | No (history) |
| `Glucose` | mg/dL | Plasma glucose 2 hours after drinking a sugar solution (**oral glucose tolerance test**) | Yes (diet, exercise, medication) |
| `BloodPressure` | mm Hg | Diastolic blood pressure (pressure between heartbeats) | Yes |
| `SkinThickness` | mm | Triceps skin-fold thickness, a rough measure of body fat | Yes |
| `Insulin` | µU/mL | Serum insulin 2 hours after the glucose drink | Partly |
| `BMI` | kg/m² | Body mass index = weight / height² | Yes |
| `DiabetesPedigreeFunction` | score (no unit) | Summarises diabetes history in relatives, weighted by how closely related they are (genetic risk) | No |
| `Age` | years | Age of the patient | No |
| **`Outcome`** (target) | 0 / 1 | 1 = diabetes diagnosed (WHO criteria), 0 = not | — |

### Diabetes Progression (regression)

442 patients. Column names are kept as scikit-learn provides them; the original paper's names are in brackets.

| Feature | Unit | Clinical meaning | Modifiable? |
|---|---|---|---|
| `age` | years | Age | No |
| `sex` | code (1 or 2) | Sex of the patient (the source does not say which code is which) | No |
| `bmi` | kg/m² | Body mass index | Yes |
| `bp` | mm Hg | Average blood pressure | Yes |
| `s1` (tc) | blood serum level\* | Total serum cholesterol | Yes |
| `s2` (ldl) | blood serum level\* | Low-density lipoprotein ("bad" cholesterol) | Yes |
| `s3` (hdl) | blood serum level\* | High-density lipoprotein ("good" cholesterol) | Yes |
| `s4` (tch) | ratio | Total cholesterol / HDL | Yes |
| `s5` (ltg) | log scale\* | Likely the log of the serum triglyceride level | Yes |
| `s6` (glu) | blood serum level\* | Blood sugar level | Yes |
| **`target`** | score (no unit) | Quantitative measure of disease progression one year after baseline | — |

\* The original source documents units only partly, so we describe these columns by meaning, not exact units. We state this limitation openly.

## A.7 Success criteria (fixed **before** looking at any results)

We write these down now so we cannot move the goalposts later. They will be checked in Section H using the **test set only**.

**Task 1: Classification (screening)**
- **Recall ≥ 0.75** on the test set: the model must catch at least 3 out of 4 diabetic patients. Recall is the main metric because missing a diabetic patient (a false negative) is worse than a false alarm, which only leads to a cheap follow-up test.
- **ROC-AUC ≥ 0.80** on the test set: the model must rank diabetic patients above non-diabetic ones well, across all thresholds.

$$\text{Recall} = \frac{TP}{TP + FN} \qquad\qquad \text{ROC-AUC} = P\big(\hat{p}(\text{random diabetic}) > \hat{p}(\text{random non-diabetic})\big)$$

where $TP$ = diabetic patients correctly flagged, $FN$ = diabetic patients missed, and $\hat{p}$ = the model's predicted probability of diabetes.

**Task 2: Regression (progression)**
- The best model must clearly beat the trivial **baseline** that always predicts the training mean (that baseline has $R^2 = 0$).
- We aim for a **test $R^2$ of about 0.40 or higher**, and we will also report RMSE in the target's own units.

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

where $y_i$ is the true progression of patient $i$, $\hat{y}_i$ the prediction, and $\bar{y}$ the mean progression. $R^2 = 1$ is perfect; $R^2 = 0$ is no better than predicting the mean.

**Explainability (both tasks)**
- The top features found by different explanation methods should broadly agree, and the direction of their effects should make clinical sense (e.g. higher glucose → higher risk).

## A.8 Load the datasets

### Pima Indians Diabetes

**What:** download the Pima data from OpenML (or read the saved copy), give the columns their standard names, turn the text label into 1/0, and save it to `data/pima.csv`.

**Why:**
- OpenML uses short column names (`preg`, `plas`, ...), so we rename them to the clearer standard names used in the feature dictionary.
- The target is stored as text (`tested_positive` / `tested_negative`); models need numbers, so we encode it as $y = 1$ if `tested_positive`, else $y = 0$.
- Saving a CSV means the notebook works offline after the first run, and everyone uses exactly the same data.

**Formula:** none in this cell (just loading and renaming).

In [2]:
PIMA_PATH = "data/pima.csv"

if os.path.exists(PIMA_PATH):
    pima = pd.read_csv(PIMA_PATH)          # offline: use the saved copy
    print("Read saved copy:", PIMA_PATH)
else:
    raw = fetch_openml(name="diabetes", version=1, as_frame=True).frame
    pima = raw.rename(columns={            # OpenML short names -> standard names
        "preg": "Pregnancies", "plas": "Glucose", "pres": "BloodPressure",
        "skin": "SkinThickness", "insu": "Insulin", "mass": "BMI",
        "pedi": "DiabetesPedigreeFunction", "age": "Age", "class": "Outcome"})
    pima["Outcome"] = (pima["Outcome"] == "tested_positive").astype(int)  # 1 = diabetic
    pima.to_csv(PIMA_PATH, index=False)
    print("Downloaded from OpenML and saved to", PIMA_PATH)

Read saved copy: data/pima.csv


### Diabetes Progression

**What:** load the progression data that ships with scikit-learn (or read the saved copy) and save it to `data/progression.csv`.

**Why:** we use `scaled=False` so the features stay in their **original clinical units** (e.g. BMI in kg/m², age in years). The default scikit-learn version is already scaled, which would hide the real values and make interpretation harder. Scaling will be done properly later, inside a pipeline (Section C).

**Formula:** none in this cell.

In [3]:
PROG_PATH = "data/progression.csv"

if os.path.exists(PROG_PATH):
    prog = pd.read_csv(PROG_PATH)          # offline: use the saved copy
    print("Read saved copy:", PROG_PATH)
else:
    # scaled=False -> keep the original clinical units
    prog = load_diabetes(as_frame=True, scaled=False).frame
    prog.to_csv(PROG_PATH, index=False)
    print("Loaded from scikit-learn and saved to", PROG_PATH)

Read saved copy: data/progression.csv


## A.9 First look at the data

### Pima: first rows and shape

**What:** show the first 5 rows and the number of rows and columns.

**Why:** to check that loading and renaming worked, and to see what a typical patient record looks like.

**Formula:** none in this cell.

In [4]:
print("Pima shape (rows, columns):", pima.shape)
pima.head()

Pima shape (rows, columns): (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### Observation
The Pima data has **768 rows and 9 columns**: 8 clinical features plus the `Outcome` target, which is now a clean 0/1 number. Even in the first five rows, `Insulin` is `0` for three patients and `SkinThickness` is `0` for one. A living person cannot have zero insulin or zero skin-fold thickness, so these zeros must mean the value was **not recorded**.

### Pima: column types and missing values

**What:** `.info()` lists each column's data type and the number of non-null (non-missing) values.

**Why:** models need numeric inputs, and we need to know whether any values are missing before we model.

**Formula:** none in this cell.

In [5]:
pima.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


### Observation
All 9 columns are numeric (7 integer, 2 float) and every column reports **768 non-null values**, so pandas thinks nothing is missing. This is misleading: the missing values are hidden as `0`, not stored as `NaN`. So `.info()` alone cannot detect them; we have to look at the actual values (next cell).

### Pima: summary statistics

**What:** `.describe()` gives the count, mean, standard deviation, minimum, quartiles and maximum of every column (transposed so each feature is one row).

**Why:** it shows each feature's typical value and spread, and reveals impossible values (like a BMI of 0).

**Formulas** (for a column with values $x_1, \dots, x_n$):

$$\text{Mean: } \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad\qquad \text{Standard deviation: } s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

- $n$ = number of values, $x_i$ = the $i$-th value, $\bar{x}$ = the mean.
- pandas divides by $n-1$ (the *sample* standard deviation), which corrects for estimating the mean from the same data.
- **Median** (the `50%` row): the middle value after sorting. Half the values are below it, half above. Unlike the mean, it is not pulled by extreme values.
- `25%` and `75%` are the first and third **quartiles** ($Q_1$, $Q_3$).

In [6]:
pima.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.85,3.37,0.00,1.00,3.00,6.00,17.00
Glucose,768.0,120.89,31.97,0.00,99.00,117.00,140.25,199.00
BloodPressure,768.0,69.11,19.36,0.00,62.00,72.00,80.00,122.00
SkinThickness,768.0,20.54,15.95,0.00,0.00,23.00,32.00,99.00
Insulin,768.0,79.80,115.24,0.00,0.00,30.50,127.25,846.00
BMI,768.0,31.99,7.88,0.00,27.30,32.00,36.60,67.10
DiabetesPedigreeFunction,768.0,0.47,0.33,0.08,0.24,0.37,0.63,2.42
Age,768.0,33.24,11.76,21.00,24.00,29.00,41.00,81.00
Outcome,768.0,0.35,0.48,0.00,0.00,0.00,1.00,1.00


### Observation
`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` all have a **minimum of 0**, which is physiologically impossible, so these are hidden missing values. For `SkinThickness` and `Insulin`, even the 25th percentile is 0, so **at least a quarter** of those values are missing. `Insulin` is strongly skewed: its mean (79.80) is far above its median (30.50), with a maximum of 846. The mean of `Outcome` is 0.35, and since the mean of a 0/1 column is the fraction of 1s, about **35% of patients are diabetic**: the classes are imbalanced.

### Progression: first rows and shape

**What:** show the first 5 rows and the shape of the progression data.

**Why:** to check that loading worked and that the values are in clinical units (not pre-scaled).

**Formula:** none in this cell.

In [7]:
print("Progression shape (rows, columns):", prog.shape)
prog.head()

Progression shape (rows, columns): (442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0,135.0


### Observation
The progression data has **442 rows and 11 columns**: 10 baseline features plus `target`. The values are in real clinical units (e.g. age 59, BMI 32.1), which confirms that `scaled=False` worked. `sex` is stored as the codes 1.0 and 2.0.

### Progression: column types and missing values

**What:** `.info()` for the progression data.

**Why:** same reason as for Pima: confirm all columns are numeric and check for missing values.

**Formula:** none in this cell.

In [8]:
prog.info()

<class 'pandas.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


### Observation
All 11 columns are numeric (`float64`) with **442 non-null values each**, so there are no missing values here. Unlike Pima, this dataset needs no imputation; scaling (Section C) will be the only preprocessing step.

### Progression: summary statistics

**What:** `.describe()` for the progression data.

**Why:** to see the range and spread of each feature and, most importantly, of the **target** we will predict.

**Formulas:** the same mean, standard deviation, median and quartile definitions as for Pima above.

In [9]:
prog.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,442.0,48.52,13.11,19.00,38.25,50.00,59.00,79.00
sex,442.0,1.47,0.50,1.00,1.00,1.00,2.00,2.00
bmi,442.0,26.38,4.42,18.00,23.20,25.70,29.28,42.20
bp,442.0,94.65,13.83,62.00,84.00,93.00,105.00,133.00
s1,442.0,189.14,34.61,97.00,164.25,186.00,209.75,301.00
s2,442.0,115.44,30.41,41.60,96.05,113.00,134.50,242.40
s3,442.0,49.79,12.93,22.00,40.25,48.00,57.75,99.00
s4,442.0,4.07,1.29,2.00,3.00,4.00,5.00,9.09
s5,442.0,4.64,0.52,3.26,4.28,4.62,5.00,6.11
s6,442.0,91.26,11.50,58.00,83.25,91.00,98.00,124.00


### Observation
The `target` ranges from **25 to 346**, with a mean of 152.13 and a standard deviation of 77.09. A model that always predicts the mean would therefore be off by roughly 77 points (in RMSE terms), and that is the error our regression models must beat. Its median (140.50) is below its mean, so the target is slightly right-skewed. There are no impossible values (e.g. the minimum BMI is 18.0). The features are on very different scales (`s5` is around 4.6 while `s1` is around 189), so distance- and coefficient-based models will need **standardisation**.

## A.10 Limitations to keep in mind

- **Narrow population:** Pima contains only adult women of Pima Indian heritage, a group with a very high diabetes rate. Results may not generalise to men, children or other populations.
- **Small datasets:** 768 and 442 patients. Scores will vary noticeably between data splits, so we always report cross-validation mean ± standard deviation, not a single number.
- **Hidden missing values:** some Pima columns use `0` where a value was not recorded (handled in Sections B and C).
- **Glucose is part of the diagnosis:** diabetes is diagnosed from glucose tests, so a strong glucose effect is expected and partly built into the label.
- **Correlation, not causation:** the models find associations in the data; they do not prove that a factor *causes* diabetes.
- **Not a diagnostic tool:** this is a decision-support prototype for a course project, not for clinical use.

## Section A summary

- We defined two tasks: **classification** of diabetes on Pima (target `Outcome`) and **regression** of disease progression (target `target`).
- We listed what each feature means clinically and which ones a patient can change.
- We fixed our success criteria in advance: **recall ≥ 0.75 and ROC-AUC ≥ 0.80** for classification; **beat the mean baseline, aiming for $R^2 \approx 0.40$ or higher** for regression.
- Both datasets are loaded and saved to `data/`. The first look already shows impossible zero values in Pima, which Section B investigates.